# 01. TOPSIS baseline (v0.1)

Ranking das 25 unidades empresa-ano com TOPSIS crisp e pesos iguais sobre a matriz v0 (6 DAs, todas critérios de benefício). Critério de pronto da semana: correlação de Spearman ≥ 0,90 com a média simples.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.data import load_matriz, DAS
from src.mcdm import rank_table

df = load_matriz()
df['unidade'] = df['empresa'] + ' ' + df['ano'].astype(str)
df[['unidade'] + DAS].head()

,unidade,DA_I,DA_II,DA_III,DA_IV,DA_V,DA_VI
0,Energisa 2021,0.75,0.00,1.00,0.75,0.75,1.0
1,Energisa 2022,0.75,0.25,0.75,0.50,0.75,1.0
2,Energisa 2023,0.50,0.50,0.75,0.75,0.50,1.0
3,Energisa 2024,0.50,0.25,0.75,0.75,0.25,1.0
4,Energisa 2025,0.50,0.25,0.75,0.50,0.75,1.0


## TOPSIS com pesos iguais

In [2]:
rk = rank_table(df, DAS, id_cols=['unidade'])
rk['media_simples'] = df.loc[rk.index, 'maturidade_media']
rk['ranking_media'] = rk['media_simples'].rank(ascending=False, method='min').astype(int)
rk[['unidade', 'topsis', 'ranking', 'media_simples', 'ranking_media']].round(3)

,unidade,topsis,ranking,media_simples,ranking_media
19,Neoenergia 2025,0.810,1,0.833,1
18,Neoenergia 2024,0.796,2,0.819,3
16,Neoenergia 2022,0.739,3,0.833,1
24,Petrobras 2025,0.664,4,0.778,4
10,Statkraft Brasil 2021,0.607,5,0.694,8
23,Petrobras 2024,0.594,6,0.750,5
12,Statkraft Brasil 2023,0.584,7,0.694,8
2,Energisa 2023,0.582,8,0.667,12
8,ENGIE Brasil 2024,0.560,9,0.750,5
21,Petrobras 2022,0.556,10,0.653,17


## Concordância com a média simples

In [3]:
rho = rk['topsis'].corr(rk['media_simples'], method='spearman')
print(f'Spearman TOPSIS x média simples: {rho:.3f}')
print('Critério (>= 0,90):', 'atendido' if rho >= 0.90 else 'NÃO atendido')
empates_media = rk['media_simples'].round(4).duplicated(keep=False).sum()
empates_topsis = rk['topsis'].round(4).duplicated(keep=False).sum()
print(f'Unidades empatadas: média simples = {empates_media}, TOPSIS = {empates_topsis}')

Spearman TOPSIS x média simples: 0.813
Critério (>= 0,90): NÃO atendido
Unidades empatadas: média simples = 17, TOPSIS = 2


**Resultado:** a correlação de Spearman com a média simples ficou abaixo de 0,90, então o critério de pronto não foi atendido com a normalização vetorial. O TOPSIS desempata a maior parte das 17 unidades que a média simples deixa empatadas (sobram 2), o que confirma o ponto do orientador sobre a baixa discriminação da v0. A célula abaixo testa a normalização min-max como análise de sensibilidade, sem escolher a que atinge a meta. Vale discutir com o orientador se 0,90 é o limiar certo para uma matriz com 6 critérios de 4 níveis.

In [4]:
rk2 = rank_table(df, DAS, id_cols=['unidade'], normalization='minmax')
rho2 = rk2['topsis'].corr(df.loc[rk2.index, 'maturidade_media'], method='spearman')
print(f'Spearman TOPSIS (min-max) x média simples: {rho2:.3f}')

Spearman TOPSIS (min-max) x média simples: 0.994


In [5]:
out = Path.cwd().parent / 'data' / 'processed' / 'ranking_v0.csv'
rk[['unidade', 'topsis', 'ranking', 'media_simples', 'ranking_media']].round(4).to_csv(out, index=False)
print('salvo em', out)

salvo em /home/claude/energia5-repo/data/processed/ranking_v0.csv
